## 0. Proof of lifeRun this first. It prints immediately, so a blank log means the run hasnot started — not that it is stuck. It also tells you whether Internetis on, which is off by default on Kaggle and breaks every install.

In [ ]:
# Immediate proof of life. Kaggle's first log lines are debugger noise; this is# the first thing that is actually yours, so it prints before anything slow.import sys, platform, subprocess, timeSTART = time.time()print("=" * 58, flush=True)print(f"  notebook started  {time.strftime('%Y-%m-%d %H:%M:%S')}", flush=True)print(f"  python {sys.version.split()[0]} on {platform.platform()}", flush=True)try:    n = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],                       capture_output=True, text=True, timeout=20).stdout.strip()    print(f"  gpu: {n or 'none (fine for harvesting)'}", flush=True)except Exception:    print("  gpu: none (fine for harvesting)", flush=True)print(f"  internet: ", end="", flush=True)try:    import urllib.request    urllib.request.urlopen("https://pypi.org", timeout=15)    print("ON", flush=True)except Exception as e:    print(f"OFF or blocked -- {type(e).__name__}. "          "Settings > Internet > On (needs a phone-verified account).", flush=True)print("=" * 58, flush=True)

# Harvest FRC match video — ColabRuns the whole pipeline (pick → download → crop → scoreboard → frames) onColab's disk, and keeps only the small, expensive-to-regenerate output onDrive: **frames, labels and the database, ~100 MB per match**. Videos stay onthe ephemeral disk and are thrown away.**Set Runtime → Change runtime type → CPU.** This needs no GPU, and asking forone burns quota you want for training.## Read this before you invest timeYouTube applies bot detection to datacenter IPs, and Colab is one. Downloadsmay work, may work for some videos, or may fail outright with *"Sign in toconfirm you're not a bot"* — and which of those you get changes month tomonth. **Cell 1 tells you in about 30 seconds.** If it fails, skip to thecookies workaround at the bottom rather than debugging the rest.

## 1. Can this session download from YouTube at all?Run this first. Everything else is wasted effort if it fails.

In [ ]:
!pip -q install -U yt-dlpimport subprocess# A Creative Commons clip from the same platform, metadata only, no download.probe = subprocess.run(    ["yt-dlp", "-J", "--no-warnings", "--no-playlist",     "https://www.youtube.com/watch?v=aqz-KE-bpKQ"],    capture_output=True, text=True, timeout=120)if probe.returncode == 0:    print("PASS - YouTube is reachable from this session. Continue.")else:    err = (probe.stderr or "").strip().splitlines()[-3:]    print("FAIL - downloads are blocked from this IP:")    print("\n".join("   " + e for e in err))    print("\n-> Use the cookies workaround at the bottom, or harvest locally.")print("  pip install finished", flush=True)

## 2. System toolsffmpeg ships with Colab; tesseract does not.

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null 2>&1!pip -q install requests numpyimport shutilfor t in ("ffmpeg", "ffprobe", "yt-dlp", "tesseract"):    print(f"  {t:10} {shutil.which(t) or 'MISSING'}")print("  apt install finished", flush=True)

## 3. The codeBuild the archive on your Mac and upload it to Drive once:```bash./deploy/make_code_archive.sh```It contains no `.env`, no data and no database — just the pipeline.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')!mkdir -p /content/work && tar -xzf /content/drive/MyDrive/tbavid_code.tgz -C /content/work%cd /content/work!ls

## 4. Your TBA keyUse Colab's secrets, not a file: click the **key icon** in the left sidebar,add `TBA_AUTH_KEY`, and enable it for this notebook. That keeps it out of thenotebook and out of anything you share.

In [ ]:
import osfrom google.colab import userdataos.environ["TBA_AUTH_KEY"] = userdata.get("TBA_AUTH_KEY")print("key loaded, length", len(os.environ["TBA_AUTH_KEY"]))

## 5. Work locally, keep only what mattersColab's own disk is fast and roomy; Drive over FUSE is slow for the hundreds ofsmall JPEG writes a match produces. So process on `/content` and sync theresults to Drive afterwards.`keep_raw` and `keep_clean` off means a match costs ~100 MB instead of ~333 MB.

In [ ]:
import json, os, pathlibos.environ["TBAVID_DATA"] = "/content/data"cfg = json.loads(pathlib.Path("config.json").read_text())cfg["keep_raw"]   = False    # sources are re-downloadablecfg["keep_clean"] = False    # cleaned videos are re-renderablepathlib.Path("config.json").write_text(json.dumps(cfg, indent=2))print({k: cfg[k] for k in ("keep_raw", "keep_clean", "sample_fps", "score_labels")})DRIVE = pathlib.Path("/content/drive/MyDrive/tbavid")(DRIVE / "frames").mkdir(parents=True, exist_ok=True)(DRIVE / "labels").mkdir(parents=True, exist_ok=True)

### Resume where you left offThe ledger is what stops you re-pulling the same match. It lives in `state/`,so restore it from Drive before pulling and save it after — otherwise every newsession starts from scratch and happily re-downloads work you already have.

In [ ]:
!mkdir -p /content/work/state!cp -f /content/drive/MyDrive/tbavid/seen.json /content/work/state/ || echo "no previous ledger - first run"!python3 run.py status | head -5

## 6. PullStart with 3 to confirm the whole chain works before committing an hour.A match takes roughly 4-6 minutes on Colab's CPU.

In [ ]:
!python3 run.py pull -n 3 --per-event-cap 2

## 7. Check it against TBA before trusting any of it

In [ ]:
!python3 run.py db sync!python3 run.py verify

## 8. Save results to Drive — do this every batchFree sessions disconnect without warning and take local disk with them.

In [ ]:
!rsync -a /content/data/frames/  /content/drive/MyDrive/tbavid/frames/!rsync -a /content/data/labels/  /content/drive/MyDrive/tbavid/labels/!cp -f /content/data/scouting.db /content/drive/MyDrive/tbavid/!cp -f /content/data/review/manifest.json /content/drive/MyDrive/tbavid/!cp -f /content/work/state/seen.json /content/drive/MyDrive/tbavid/!du -sh /content/drive/MyDrive/tbavid/*

## 9. Bigger batchesWith the chain proven, raise the count. Keep each run inside the session limit— roughly 5 minutes a match, so 20 matches is about 1.5-2 hours. Re-run cell 8afterwards, every time.

In [ ]:
!python3 run.py pull -n 20 --per-event-cap 2

---## If cell 1 failed: the cookies workaroundExport cookies from a browser where you are signed in to YouTube, which makesthe request look like a logged-in user rather than an anonymous datacenter.1. Install a `cookies.txt` browser extension (Netscape format).2. Visit youtube.com while signed in and export.3. Upload `cookies.txt` to Drive.4. Run the cell below, then retry cell 1.Caveats worth knowing: cookies expire and need re-exporting; you are attachingyour account to this traffic, so use an account you do not mind risking; andthis may simply stop working. If it does, harvest on your Mac — a home IP isthe thing Colab cannot give you — and use Colab only for training.

In [ ]:
import pathlib, jsonsrc = pathlib.Path("/content/drive/MyDrive/cookies.txt")if src.exists():    dst = pathlib.Path("/content/work/cookies.txt"); dst.write_bytes(src.read_bytes())    cfg = json.loads(pathlib.Path("/content/work/config.json").read_text())    cfg["ytdlp_cookies"] = str(dst)    pathlib.Path("/content/work/config.json").write_text(json.dumps(cfg, indent=2))    print("cookies wired into config.json ->", dst)else:    print("put cookies.txt at the top level of My Drive first")